In [0]:
%run ./utils/validations

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "pb_threat_intelligence")

run_id = dbutils.widgets.get("run_id").strip()
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
if not run_id:
    raise ValueError("An explicit run_id is required")

audit_table = f"{catalog}.{schema}.feed_count_audit"
output_table = f"{catalog}.{schema}.domains_classification_current"

In [0]:
current = spark.table(output_table)
if current.filter(~F.col("run_id").eqNullSafe(F.lit(run_id))).limit(1).count():
    raise ValueError("Current table belongs to another run")
validate_current_state(current)
validate_count_balance(audit_table, run_id, current.count())